In [14]:
import wikipediaapi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document
import re
import os
from datetime import datetime

### Langkah 1: Setup Model dan Parameter

In [ ]:
print("=== SETUP MODEL DAN PARAMETER ===")
print("1. Inisialisasi model embedding khusus bahasa Indonesia...")
embedding_model = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

print("2. Konfigurasi LLM untuk generasi jawaban...")
llm = ChatOpenAI(
    api_key=os.getenv("GROQ_API_KEY"), 
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile",
    max_tokens=512,
    temperature=0.3
)

print("3. Membuat template prompt untuk mengurangi halusinasi...")
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
    Anda adalah asisten sejarah Indonesia. Jawablah pertanyaan HANYA berdasarkan konteks di bawah ini.
    Jika tidak ada informasi yang relevan, katakan 'Data tidak ditemukan'.
    
    Konteks:
    {context}
    
    Pertanyaan: {question}
    
    Jawaban:
    """
)


=== SETUP MODEL DAN PARAMETER ===
1. Inisialisasi model embedding khusus bahasa Indonesia...
2. Konfigurasi LLM untuk generasi jawaban...
3. Membuat template prompt untuk mengurangi halusinasi...


### Langkah 2: Mengumpulkan Data Pemilu dari Wikipedia

In [ ]:
print("\n=== MENGUMPULKAN DATA SEJARAH iNDONESIA ===")
wiki = wikipediaapi.Wikipedia(
    user_agent='HistoryApp/1.0',
    language='id'
)

history_pages = ['Sejarah Indonesia', 'Sejarah Indonesia (1945–1949)', 'Soekarno', 'Pertempuran Surabaya', 'Pemerintahan Sipil Hindia Belanda']

documents = []
current_year = datetime.now().year

print("Memproses halaman Wikipedia...")
for page_title in history_pages:
    print(f"\n- Mengambil: {page_title}")
    page = wiki.page(page_title)
    
    if page.exists():
        years_in_text = re.findall(r'\b(19\d{2}|20\d{2})\b', page.text)
        min_year = min(years_in_text) if years_in_text else "1945"
        max_year = max(years_in_text) if years_in_text else str(current_year)
        
        print(f"  Rentang tahun: {min_year}-{max_year}")
        print(f"  Ukuran teks: {len(page.text)} karakter")
        

        print("  Membagi teks menjadi chunk...")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", ". ", "? ", "! "]
        )
        
        chunks = text_splitter.split_text(page.text)
        print(f"  Jumlah chunk: {len(chunks)}")
        
        for chunk in chunks:
            documents.append(Document(
                page_content=chunk,
                metadata={
                    "source": page_title,
                    "url": page.fullurl,
                    "min_year": min_year,
                    "max_year": max_year
                }
            ))
    else:
        print(f"  Halaman tidak ditemukan!")

print(f"\nTotal dokumen terkumpul: {len(documents)}")


=== MENGUMPULKAN DATA SEJARAH iNDONESIA ===
Memproses halaman Wikipedia...

- Mengambil: Sejarah Indonesia
  Rentang tahun: 1901-2020
  Ukuran teks: 55296 karakter
  Membagi teks menjadi chunk...
  Jumlah chunk: 176

- Mengambil: Sejarah Indonesia (1945–1949)
  Rentang tahun: 1942-1949
  Ukuran teks: 33821 karakter
  Membagi teks menjadi chunk...
  Jumlah chunk: 108

- Mengambil: Soekarno
  Rentang tahun: 1901-2013
  Ukuran teks: 92376 karakter
  Membagi teks menjadi chunk...
  Jumlah chunk: 301

- Mengambil: Pertempuran Surabaya
  Rentang tahun: 1942-2016
  Ukuran teks: 15777 karakter
  Membagi teks menjadi chunk...
  Jumlah chunk: 53

- Mengambil: Pemerintahan Sipil Hindia Belanda
  Rentang tahun: 1944-2005
  Ukuran teks: 4343 karakter
  Membagi teks menjadi chunk...
  Jumlah chunk: 14

Total dokumen terkumpul: 652


### Langkah 3: Membuat Vector Store

In [ ]:
print("\n=== MEMBUAT BASIS PENGETAHUAN ===")
print("Membuat vektor store dengan metadata...")
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

print("Menyimpan ke lokal...")
vector_store.save_local("history_faiss_index")
print("Knowledge base berhasil dibuat!")


=== MEMBUAT BASIS PENGETAHUAN ===
Membuat vektor store dengan metadata...
Menyimpan ke lokal...
Knowledge base berhasil dibuat!


### Langkah 4: Inisialisasi Sistem Tanya Jawab

In [ ]:

print("\n=== INISIALISASI SISTEM TANYA JAWAB ===")
print("Memuat knowledge base...")
vector_store = FAISS.load_local(
    "history_faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("Membuat retriever dengan threshold relevansi...")
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": lambda meta: int(meta.get("min_year", 0)) > 2000 
    },
    verbose=False
)


print("Menyiapkan RAG chain...")
qa_system = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)


=== INISIALISASI SISTEM TANYA JAWAB ===
Memuat knowledge base...
Membuat retriever dengan threshold relevansi...
Menyiapkan RAG chain...


### Langkah 5: Menjalankan Kueri Contoh

In [ ]:

print("\n=== MENJALANKAN KUERI CONTOH ===")
queries = [
    "Apa peran Soekarno dalam kemerdekaan Indonesia?",
    "Kapan proklamasi kemerdekaan diumumkan?",
    "Jelaskan sejarah NICA"
]

for query in queries:
    print(f"\nPERTANYAAN: {query}")
    response = qa_system({"query": query})
    print(f"JAWABAN: {response['result']}")
    


=== MENJALANKAN KUERI CONTOH ===

PERTANYAAN: Apa peran Soekarno dalam kemerdekaan Indonesia?
JAWABAN: Soekarno memainkan peran penting dalam kemerdekaan Indonesia. Ia adalah salah satu tokoh utama yang memperjuangkan kemerdekaan Indonesia dari penjajahan Belanda. Bersama dengan Mohammad Hatta, Soekarno menjadi proklamator kemerdekaan Indonesia pada tanggal 17 Agustus 1945. Ia juga menjadi presiden pertama Republik Indonesia dan memimpin negara ini dalam membangun fondasi dan struktur pemerintahan pasca-kemerdekaan.

PERTANYAAN: Kapan proklamasi kemerdekaan diumumkan?
JAWABAN: Proklamasi kemerdekaan Indonesia diumumkan pada tanggal 17 Agustus 1945.

PERTANYAAN: Jelaskan sejarah NICA
JAWABAN: NICA (Nederlandsch-Indische Civiele Administratie) atau Administrasi Sipil Hindia Belanda adalah sebuah badan pemerintahan sementara yang dibentuk oleh pemerintah Belanda pada tahun 1946 untuk mengelola wilayah-wilayah di Indonesia yang telah direbut kembali dari Jepang setelah Proklamasi Kemerdek

### Langkah 6: Memperbarui Data dengan Informasi Baru

In [ ]:

print("\n=== MEMPERBARUI DATA DENGAN INFORMASI BARU ===")
new_data = {
    "Peristiwa Bandung Lautan Api": 
        "Bandung Lautan Api adalah peristiwa kebakaran besar yang terjadi di kota Bandung pada 23 Maret 1946. "
        "Peristiwa ini terjadi sebagai upaya para pejuang kemerdekaan Indonesia untuk mencegah tentara Sekutu "
        "dan tentara NICA Belanda menggunakan kota Bandung sebagai markas militer. Atas peristiwa ini, "
        "lagu 'Halo-Halo Bandung' menjadi terkenal."
}

print("Membuat dokumen baru...")
current_year = datetime.now().year
new_docs = []

for title, content in new_data.items():
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    
    chunks = text_splitter.split_text(content)
    for chunk in chunks:
        new_docs.append(Document(
            page_content=chunk,
            metadata={
                "source": title,
                "url": "https://id.wikipedia.org/wiki/Bandung_Lautan_Api",
                "min_year": "1946",
                "max_year": "1946"
            }
        ))

print(f"Menambahkan {len(new_docs)} dokumen baru...")
vector_store.add_documents(new_docs)
vector_store.save_local("history_faiss_index")
print("Basis pengetahuan berhasil diperbarui!")

# Langkah 9: Menguji Data Baru
print("\n=== MENGUJI DATA BARU ===")
query = "Apa itu Bandung Lautan Api?"
print(f"PERTANYAAN: {query}")

response = qa_system({"query": query})
clean_response = response['result']
print(f"JAWABAN: {clean_response}")


=== MEMPERBARUI DATA DENGAN INFORMASI BARU ===
Membuat dokumen baru...
Menambahkan 1 dokumen baru...
Basis pengetahuan berhasil diperbarui!

=== MENGUJI DATA BARU ===
PERTANYAAN: Apa itu Bandung Lautan Api?
JAWABAN: Bandung Lautan Api adalah sebuah peristiwa sejarah yang terjadi pada tanggal 23 Maret 1946, ketika kota Bandung dibakar oleh penduduknya sendiri sebagai bentuk perlawanan terhadap pasukan Inggris yang akan menduduki kota tersebut. Peristiwa ini dilakukan untuk mencegah pasukan Inggris menggunakan kota Bandung sebagai basis militer dan untuk mempertahankan kemerdekaan Indonesia.


In [44]:
new_docs

[Document(metadata={'source': 'Peristiwa Bandung Lautan Api', 'url': 'https://id.wikipedia.org/wiki/Bandung_Lautan_Api', 'min_year': '1946', 'max_year': '1946'}, page_content="Bandung Lautan Api adalah peristiwa kebakaran besar yang terjadi di kota Bandung pada 23 Maret 1946. Peristiwa ini terjadi sebagai upaya para pejuang kemerdekaan Indonesia untuk mencegah tentara Sekutu dan tentara NICA Belanda menggunakan kota Bandung sebagai markas militer. Atas peristiwa ini, lagu 'Halo-Halo Bandung' menjadi terkenal.")]

#### Analisis Aplikasi RAG untuk Sejarah Indonesia

1. Mengapa Menggunakan LLM (Llama-3.3-70B-Versatile)?
Model Llama-3-70B memiliki kemampuan pemahaman bahasa Indonesia yang baik karena dilatih pada dataset multibahasa. Dengan 70B parameter, model dapat menangani konteks sejarah yang kompleks dan hubungan antar peristiwa.

2. Mengapa Memilih Model Embedding (Qwen/Qwen3-Embedding-0.6B)?
Model 0.6B parameter menangkap nuansa semantik teks sejarah dengan baik.

3. Mengapa Memilih Vector DB (FAISS)?
Kompresi vektor (dengan normalisasi) menghemat memori. Didesain khusus untuk pencarian vektor berdimensi tinggi.

4. Kekurangan Aplikasi RAG yang Dikembangkan:
- Tidak ada mekanisme feedback
- Tidak ada mekanisme evaluasi akurasi jawaban
- Tidak ada tracking untuk pertanyaan yang gagal dijawab
- Tidak ada ekstraksi entitas khusus (tokoh, lokasi, peristiwa)
